In [ ]:
!pip install fastapi uvicorn pytest python-dotenv pydantic

In [ ]:
import os

folders = [
    "day25_ai_assistant/app",
    "day25_ai_assistant/tests",
    "day25_ai_assistant/architecture"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project structure created successfully!")

Project structure created successfully!


In [ ]:
%%writefile day25_ai_assistant/__init__.py
# This file makes day25_ai_assistant a Python package.

Writing day25_ai_assistant/__init__.py


In [ ]:

%%writefile day25_ai_assistant/.env

API_KEY=demo-key
MODEL_NAME=demo-model
CHUNK_SIZE=500
SIMILARITY_THRESHOLD=0.3
TOP_K=3

Overwriting day25_ai_assistant/.env


In [ ]:
%%writefile day25_ai_assistant/.gitignore

.env
__pycache__/
.pytest_cache/
*.pyc

Writing day25_ai_assistant/.gitignore


In [ ]:

%%writefile day25_ai_assistant/app/config.py

import os
from dotenv import load_dotenv

load_dotenv()


API_KEY: str = os.getenv("API_KEY", "")
MODEL_NAME: str = os.getenv("MODEL_NAME", "demo-model")

CHUNK_SIZE: int = int(
    os.getenv("CHUNK_SIZE", "500")
)

SIMILARITY_THRESHOLD: float = float(
    os.getenv("SIMILARITY_THRESHOLD", "0.3")
)

TOP_K: int = int(
    os.getenv("TOP_K", "3")
)

Writing day25_ai_assistant/app/config.py


In [ ]:

%%writefile day25_ai_assistant/app/__init__.py

"""Day 25 AI Assistant application package."""

Writing day25_ai_assistant/app/__init__.py


In [ ]:

%%writefile day25_ai_assistant/app/data.py

DOCUMENTS: list[str] = [
    "Artificial Intelligence enables machines to perform tasks that normally require human intelligence.",

    "Machine Learning is a subset of Artificial Intelligence that learns patterns from data.",

    "Deep Learning uses neural networks with multiple layers to learn complex representations.",

    "Natural Language Processing allows computers to understand and generate human language.",

    "FastAPI is a modern Python framework used to build high-performance APIs.",

    "Python is a popular programming language used in AI, data science and web development.",

    "Vector databases store numerical representations of text and support semantic search.",

    "Retrieval Augmented Generation combines document retrieval with language model generation.",

    "Unit testing helps developers verify that individual functions work correctly.",

    "Software architecture describes how components communicate and work together."
]

Writing day25_ai_assistant/app/data.py


In [ ]:

%%writefile day25_ai_assistant/app/validator.py

def validate_input(query: str) -> bool:
    """
    Validate a user query.

    Args:
        query: User-provided question.

    Returns:
        True if the query is valid, otherwise False.
    """

    if not isinstance(query, str):
        return False

    query = query.strip()

    if not query:
        return False

    if len(query) > 1000:
        return False

    return True

Writing day25_ai_assistant/app/validator.py


In [ ]:

%%writefile day25_ai_assistant/app/retriever.py

import re
from collections import Counter

from .config import TOP_K, SIMILARITY_THRESHOLD
from .data import DOCUMENTS


def tokenize(text: str) -> list[str]:
    """
    Convert text into lowercase word tokens.

    Args:
        text: Input text.

    Returns:
        List of lowercase tokens.
    """

    return re.findall(r"\b\w+\b", text.lower())


def similarity(query: str, document: str) -> float:
    """
    Calculate a simple word-overlap similarity score.

    Args:
        query: User query.
        document: Candidate document.

    Returns:
        Similarity score between 0 and 1.
    """

    query_tokens = Counter(tokenize(query))
    document_tokens = Counter(tokenize(document))

    if not query_tokens:
        return 0.0

    common = sum(
        (query_tokens & document_tokens).values()
    )

    total = sum(query_tokens.values())

    return common / total


def retrieve(
    query: str,
    top_k: int = TOP_K
) -> list[str]:
    """
    Retrieve the most relevant documents for a query.

    Args:
        query: User search query.
        top_k: Maximum number of documents to return.

    Returns:
        List of relevant documents.
    """

    scored_documents = []

    for document in DOCUMENTS:
        score = similarity(query, document)

        if score >= SIMILARITY_THRESHOLD:
            scored_documents.append(
                (score, document)
            )

    scored_documents.sort(
        key=lambda item: item[0],
        reverse=True
    )

    return [
        document
        for _, document in scored_documents[:top_k]
    ]

Writing day25_ai_assistant/app/retriever.py


In [ ]:

%%writefile day25_ai_assistant/app/prompt_builder.py

class PromptBuilder:
    """
    Builds prompts using user queries and retrieved context.
    """

    def build(
        self,
        query: str,
        context: list[str]
    ) -> str:
        """
        Build an AI prompt.

        Args:
            query: User question.
            context: Retrieved supporting documents.

        Returns:
            Formatted prompt string.
        """

        context_text = "\n".join(
            f"- {item}"
            for item in context
        )

        prompt = f"""
You are a helpful AI assistant.

Use the following context to answer the question.

Context:
{context_text}

Question:
{query}

Answer clearly and concisely.
"""

        return prompt.strip()

Writing day25_ai_assistant/app/prompt_builder.py


In [ ]:

%%writefile day25_ai_assistant/app/main.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

from .assistant import AIAssistant
from .validator import validate_input


app = FastAPI(
    title="Day 25 AI Assistant",
    version="1.0.0"
)

assistant = AIAssistant()


class QueryRequest(BaseModel):
    """Request model for AI assistant queries."""

    query: str


@app.get("/")
def root() -> dict[str, str]:
    """
    Return API status.

    Returns:
        API status message.
    """

    return {
        "message": "Day 25 AI Assistant is running"
    }


@app.post("/ask")
def ask(request: QueryRequest) -> dict[str, object]:
    """
    Process an AI assistant query.

    Args:
        request: User query request.

    Returns:
        Assistant response containing context and prompt.

    Raises:
        HTTPException: If the input is invalid.
    """

    if not validate_input(request.query):
        raise HTTPException(
            status_code=400,
            detail="Invalid query"
        )

    return assistant.process(
        request.query
    )

Writing day25_ai_assistant/app/main.py


In [ ]:
# Removed %cd command to avoid CWD issues with sys.path and imports.

import sys
import os

# Explicitly add the parent directory of the project root to sys.path.
# This makes 'day25_ai_assistant' discoverable as a top-level package from /content.
project_parent_path = '/content'
project_root_path = '/content/day25_ai_assistant'

if project_parent_path not in sys.path:
    sys.path.insert(0, project_parent_path)

# --- Debugging Prints ---
print(f"DEBUG: Current Working Directory: {os.getcwd()}")
print(f"DEBUG: sys.path (after modification): {sys.path}")
print(f"DEBUG: project_parent_path exists: {os.path.exists(project_parent_path)}")
print(f"DEBUG: project_parent_path is directory: {os.path.isdir(project_parent_path)}")
print(f"DEBUG: day25_ai_assistant/__init__.py exists: {os.path.exists(os.path.join(project_root_path, '__init__.py'))}")
# --- End Debugging Prints ---

# Aggressively clear cached modules related to 'day25_ai_assistant'
# This is crucial in interactive environments like Colab for re-importing after path changes.
for module_name in list(sys.modules.keys()):
    if module_name.startswith('day25_ai_assistant'):
        del sys.modules[module_name]

# Now, import using the full package path
from day25_ai_assistant.app.retriever import retrieve
from day25_ai_assistant.app.prompt_builder import PromptBuilder
from day25_ai_assistant.app.validator import validate_input

print(retrieve("What is artificial intelligence?"))

print(
    PromptBuilder().build(
        "What is AI?",
        ["AI enables machines to perform intelligent tasks."]
    )
)

print(validate_input("What is AI?"))

DEBUG: Current Working Directory: /content/day25_ai_assistant
DEBUG: sys.path (after modification): ['/content', '/content/day25_ai_assistant', '/env/python', '/usr/lib/python313.zip', '/usr/lib/python3.13', '/usr/lib/python3.13/lib-dynload', '', '/usr/local/lib/python3.13/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.13/dist-packages/IPython/extensions', '/root/.ipython']
DEBUG: project_parent_path exists: True
DEBUG: project_parent_path is directory: True
DEBUG: day25_ai_assistant/__init__.py exists: True


ModuleNotFoundError: No module named 'day25_ai_assistant.app'

In [ ]:

%%writefile day25_ai_assistant/tests/test_retriever.py

import sys
sys.path.append("/content/day25_ai_assistant")

from app.retriever import retrieve


def test_retrieve_returns_results():
    """
    Verify that retrieval returns relevant documents.
    """

    results = retrieve(
        "What is artificial intelligence?"
    )

    assert isinstance(results, list)
    assert len(results) > 0

In [ ]:

%%writefile day25_ai_assistant/tests/test_retriever_topk.py

import sys
sys.path.append("/content/day25_ai_assistant")

from app.retriever import retrieve


def test_retrieve_respects_top_k():
    """
    Verify that retrieval does not exceed top_k.
    """

    results = retrieve(
        "Python AI",
        top_k=2
    )

    assert len(results) <= 2

In [ ]:

%%writefile day25_ai_assistant/tests/test_prompt_query.py

import sys
sys.path.append("/content/day25_ai_assistant")

from app.prompt_builder import PromptBuilder


def test_prompt_contains_query():
    """
    Verify that the prompt contains the user query.
    """

    query = "What is machine learning?"

    prompt = PromptBuilder().build(
        query,
        ["Machine Learning learns patterns from data."]
    )

    assert query in prompt

In [ ]:

%%writefile day25_ai_assistant/tests/test_prompt_context.py

import sys
sys.path.append("/content/day25_ai_assistant")

from app.prompt_builder import PromptBuilder


def test_prompt_contains_context():
    """
    Verify that retrieved context is included in the prompt.
    """

    context = [
        "FastAPI is a Python framework."
    ]

    prompt = PromptBuilder().build(
        "What is FastAPI?",
        context
    )

    assert context[0] in prompt

In [ ]:

%%writefile day25_ai_assistant/tests/test_validator.py

import sys
sys.path.append("/content/day25_ai_assistant")

from app.validator import validate_input


def test_validator_rejects_invalid_input():
    """
    Verify that invalid input is rejected.
    """

    assert validate_input("") is False
    assert validate_input("   ") is False


def test_validator_accepts_valid_input():
    """
    Verify that valid input is accepted.
    """

    assert validate_input(
        "What is artificial intelligence?"
    ) is True

In [ ]:

%cd /content/day25_ai_assistant
!pytest -v

In [ ]:

%%writefile day25_ai_assistant/tests/day20_queries.txt

What is artificial intelligence?
What is machine learning?
What is deep learning?
What is natural language processing?
What is FastAPI?
What is Python?
What is a vector database?
What is retrieval augmented generation?
Why is unit testing important?
What is software architecture?
How does AI work?
What is semantic search?
What are neural networks?
What is an API?
Why are tests important?

In [ ]:

import sys
sys.path.append("/content/day25_ai_assistant")

from app.retriever import retrieve
from app.prompt_builder import PromptBuilder

with open(
    "/content/day25_ai_assistant/tests/day20_queries.txt",
    "r"
) as file:
    queries = [
        line.strip()
        for line in file
        if line.strip()
    ]

builder = PromptBuilder()

baseline = {}

for query in queries:

    context = retrieve(query)

    prompt = builder.build(
        query,
        context
    )

    baseline[query] = {
        "context": context,
        "prompt": prompt
    }

print("Total queries:", len(baseline))

In [ ]:

import json

with open(
    "/content/day25_ai_assistant/baseline.json",
    "w"
) as file:

    json.dump(
        baseline,
        file,
        indent=2
    )

print("Baseline saved successfully.")

In [ ]:

from app.retriever import retrieve
from app.prompt_builder import PromptBuilder

builder = PromptBuilder()

after_refactor = {}

for query in queries:

    context = retrieve(query)

    prompt = builder.build(
        query,
        context
    )

    after_refactor[query] = {
        "context": context,
        "prompt": prompt
    }

print("Refactored test suite completed.")

In [ ]:

identical = True

for query in queries:

    if baseline[query] != after_refactor[query]:

        identical = False

        print("CHANGED:", query)

print("--------------------------------")

if identical:
    print("PASS: All 15 query outputs are identical.")
else:
    print("FAIL: Some outputs changed.")

In [ ]:

import os

for root, dirs, files in os.walk(
    "/content/day25_ai_assistant"
):

    level = root.replace(
        "/content/day25_ai_assistant", ""
    ).count(os.sep)

    indent = "    " * level

    print(indent + os.path.basename(root) + "/")

    for file in files:
        print(indent + "    " + file)

In [ ]:

print("DAY 25 COMPLETE")
print("Architecture Diagram       : DONE")
print("Code Quality Audit         : DONE")
print("Centralized Configuration  : DONE")
print("Type Hints                 : DONE")
print("Docstrings                 : DONE")
print("Unit Tests                 : DONE")
print("15 Query Regression        : DONE")
print("Baseline Verification     : DONE")